# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking/scoring task. The decision is straightforward: given limited reviewer capacity, which visible pages should be audited first?

The product decision is a repeatable yes/no review policy at the page level. I encode that policy as an explicit proxy label (`opportunity_label`) using clear criteria (volume floor + tier-adjusted CTR gap), then test whether the ranking surfaces more positives in the top-K queue. A supervised framing gives me a clear objective, measurable error trade-offs (false positives waste review time; false negatives miss opportunity), and a disciplined way to tune thresholds to reviewer budget. In this notebook, I implement a transparent rule-based baseline that is consistent with this supervised setup.

In [1]:
import os
import pathlib

import pandas as pd


def find_repo_root(start=None):
    start = pathlib.Path(start or os.getcwd())
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root')

repo_root = find_repo_root()
os.chdir(repo_root)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
volume_floor = 500
gap_cutoff = -0.5

print(f'Rows in starter slice: {len(df):,}')
print(f"Unique clients: {df['client_id'].nunique()}")
print(f'Provisional supervision thresholds: volume_floor={volume_floor}, relative_gap_cutoff={gap_cutoff}')

Rows in starter slice: 30,000
Unique clients: 32
Provisional supervision thresholds: volume_floor=500, relative_gap_cutoff=-0.5


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a provisional proxy label: `1` for pages that are high-volume and materially below their pooled CTR benchmark, `0` otherwise.

This is a policy label built from current-window signals, not a ground-truth future outcome. I use it deliberately for queue framing because it is explicit, reproducible, and easy to audit. The trade-off is clear: false positives spend reviewer time on weak candidates, while false negatives miss pages with meaningful upside.

In [2]:
import numpy as np

position_tiers = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
has_position_data = df[(df['impressions_90d'] >= 100) & (df['avg_position'] > 0)].copy()

# Use pooled CTR: total clicks / total impressions within each position tier.
tier_benchmarks = {}
for tier in position_tiers:
    tier_data = has_position_data[has_position_data['position_tier'] == tier]
    impressions = tier_data['impressions_90d'].sum()
    if impressions > 0:
        tier_benchmarks[tier] = tier_data['clicks_90d'].sum() / impressions * 100

has_position_data['expected_ctr'] = has_position_data['position_tier'].map(tier_benchmarks)
has_position_data['relative_ctr_gap'] = np.where(
    has_position_data['expected_ctr'].fillna(0) > 0,
    (has_position_data['ctr'] - has_position_data['expected_ctr']) / has_position_data['expected_ctr'],
    np.nan,
)
has_position_data['opportunity_label'] = (
    (has_position_data['impressions_90d'] >= volume_floor) &
    (has_position_data['relative_ctr_gap'] <= gap_cutoff)
).astype(int)

label_counts = has_position_data['opportunity_label'].value_counts().sort_index()
print('Label counts:')
print(label_counts)
print('\nPooled tier benchmarks:')
print(pd.Series(tier_benchmarks, name='pooled_ctr').to_string())
print('\nExamples of positive labels:')
print(has_position_data[has_position_data['opportunity_label'] == 1][['content_id', 'position_tier', 'impressions_90d', 'ctr', 'expected_ctr', 'relative_ctr_gap']].head(5).to_string(index=False))

Label counts:
opportunity_label
0    14631
1     7375
Name: count, dtype: int64

Pooled tier benchmarks:
top_3       0.487133
page_1      0.349854
striking    0.346828
page_3_5    0.154719
deep        0.039482

Examples of positive labels:
          content_id position_tier  impressions_90d  ctr  expected_ctr  relative_ctr_gap
content_a1fb4e703a9e      page_3_5            15320 0.05      0.154719         -0.676834
content_d4084a4bc775        page_1             3970 0.03      0.349854         -0.914250
content_a63219c6e95a      page_3_5             1724 0.06      0.154719         -0.612201
content_c27558df2b0c        page_1             1240 0.16      0.349854         -0.542666
content_78bd1d4a1d4d        page_1            13848 0.15      0.349854         -0.571250


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The success metric should be ranking-focused and reviewer-budget aware. I defend Precision@100 because the operational decision is 'which 100 pages should I inspect first?' I also report Precision@20 and Precision@50, plus sensitivity to the gap threshold, so policy choices are explicit rather than arbitrary.

In [3]:
underperf_gap = (-has_position_data['relative_ctr_gap']).clip(lower=0)
rank_score = has_position_data['impressions_90d'] * underperf_gap
ranked = has_position_data.assign(score=rank_score).sort_values(['score', 'impressions_90d'], ascending=[False, False])

print('Precision@K:')
for k in [20, 50, 100]:
    topk = ranked.head(k)
    precision_at_k = topk['opportunity_label'].mean()
    print(f"  Precision@{k}: {precision_at_k:.3f} ({topk['opportunity_label'].sum()}/{len(topk)})")

print('\nSensitivity to gap cutoff (K=100):')
top100_index = ranked.head(100).index
for cutoff in [-0.3, -0.5, -0.7]:
    proxy_label = ((has_position_data['impressions_90d'] >= volume_floor) & (has_position_data['relative_ctr_gap'] <= cutoff)).astype(int)
    positives = int(proxy_label.loc[top100_index].sum())
    precision_at_100 = proxy_label.loc[top100_index].mean()
    print(f"  cutoff={cutoff:+.1f}: Precision@100 = {precision_at_100:.3f} ({positives}/100)")

Precision@K:
  Precision@20: 0.800 (16/20)
  Precision@50: 0.880 (44/50)
  Precision@100: 0.830 (83/100)

Sensitivity to gap cutoff (K=100):
  cutoff=-0.3: Precision@100 = 0.970 (97/100)
  cutoff=-0.5: Precision@100 = 0.830 (83/100)
  cutoff=-0.7: Precision@100 = 0.600 (60/100)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content page. Each row in the slice represents one page/content item, with historical signal columns from the starter dataset. The prediction task is at the page level, not at the client level or the query level.

In [4]:
analysis_slice = has_position_data[['content_id', 'client_id', 'position_tier', 'impressions_90d', 'ctr', 'engagement_rate', 'avg_position', 'expected_ctr', 'relative_ctr_gap', 'opportunity_label']].copy()
print('Rows in analysis slice:', len(analysis_slice))
print('Columns:')
print(analysis_slice.columns.tolist())
print('\nSample rows:')
print(analysis_slice.head(5).to_string(index=False))

Rows in analysis slice: 22006
Columns:
['content_id', 'client_id', 'position_tier', 'impressions_90d', 'ctr', 'engagement_rate', 'avg_position', 'expected_ctr', 'relative_ctr_gap', 'opportunity_label']

Sample rows:
          content_id         client_id position_tier  impressions_90d  ctr  engagement_rate  avg_position  expected_ctr  relative_ctr_gap  opportunity_label
content_304f48230142 client_f369cb89fc      striking             3803 0.76             5.88          10.6      0.346828          1.191285                  0
content_a1fb4e703a9e client_4e07408562      page_3_5            15320 0.05             0.00          20.3      0.154719         -0.676834                  1
content_9aa793d4d895 client_7f2253d7e2      page_3_5            12581 0.09             0.00          36.5      0.154719         -0.418302                  0
content_331d6c4de07b client_19581e27de        page_1            11751 0.49             1.28           6.2      0.349854          0.400584                  0

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like 'flag all pages with CTR below 0.1%' is too blunt. Pooled CTR differs by position tier, and low-volume pages add substantial noise. A tier-adjusted ranking setup handles this heterogeneity and produces a budget-aligned review queue instead of a one-threshold list.

In [5]:
tier_summary = has_position_data.groupby('position_tier').agg(
    n_pages=('content_id', 'count'),
    clicks=('clicks_90d', 'sum'),
    impressions=('impressions_90d', 'sum'),
    median_ctr=('ctr', 'median'),
    median_impressions=('impressions_90d', 'median')
)
tier_summary['pooled_ctr'] = tier_summary['clicks'] / tier_summary['impressions'] * 100
print('Tier benchmarks:')
print(tier_summary.to_string())
print('\nPooled CTR is the benchmark used by this notebook; median CTR is retained for comparison.')
for tier, row in tier_summary.iterrows():
    print(f"{tier}: pooled CTR={row['pooled_ctr']:.2f}%, median CTR={row['median_ctr']:.2f}%, median impressions={row['median_impressions']:.0f}")

Tier benchmarks:
               n_pages  clicks  impressions  median_ctr  median_impressions  pooled_ctr
position_tier                                                                          
deep               879     479      1213203        0.00               426.0    0.039482
page_1            8633  313097     89493618        0.23              2945.0    0.349854
page_3_5          6058   54370     35141017        0.06              1210.5    0.154719
striking          5903   79584     22946217        0.15              1388.0    0.346828
top_3              533   34222      7025180        0.19              2918.0    0.487133

Pooled CTR is the benchmark used by this notebook; median CTR is retained for comparison.
deep: pooled CTR=0.04%, median CTR=0.00%, median impressions=426
page_1: pooled CTR=0.35%, median CTR=0.23%, median impressions=2945
page_3_5: pooled CTR=0.15%, median CTR=0.06%, median impressions=1210
striking: pooled CTR=0.35%, median CTR=0.15%, median impressions=1388
top

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.